# Qwen RAG Streamlit UI (Google Colab)

This is `rag_qwen_streamlit.py` as a Colab notebook. Run the cells in order. The last cell starts Streamlit and opens the UI.

- **Embeddings:** `Qwen/Qwen3-Embedding-0.6B`
- **LLM:** Qwen2.5 Instruct (1.5B on GPU, 0.5B on CPU)
- **Docs:** Google Drive (`MyDrive/langchain_rag/docs`)
- **Chroma index:** local Colab disk (`/content/chroma_db_qwen3_embed`). Do not put Chroma on Drive — Drive FUSE breaks Chroma's SQLite/Rust client (`RustBindingsAPI` / `default_tenant` errors).

**Before you run:** Runtime → Change runtime type → T4 GPU. Runtime → Run all. If you already hit the Chroma error, re-run from the install cell after this update.

In [ ]:
%pip install -q pysqlite3-binary streamlit langchain-chroma "chromadb>=0.5.23,<1.1.0" langchain-huggingface langchain-core langchain-community langchain-text-splitters "sentence-transformers>=2.7.0" "transformers>=4.51.0" accelerate

import os
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import pysqlite3

    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DOCS_DIR = Path("/content/drive/MyDrive/langchain_rag/docs")
    print("Docs persist on Google Drive.")
except Exception:
    DOCS_DIR = Path("/content/langchain_rag/docs")
    print("Google Drive not available; docs are local to this session.")

# Chroma must live on local Colab disk, not Drive.
CHROMA_DIR = Path("/content/chroma_db_qwen3_embed")
DOCS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

os.environ["RAG_DOCS_DIR"] = str(DOCS_DIR)
os.environ["RAG_CHROMA_DIR"] = str(CHROMA_DIR)
print("Docs:", DOCS_DIR)
print("Chroma:", CHROMA_DIR)

In [ ]:
samples = {
    "crashloopbackoff.txt": (
        "Kubernetes CrashLoopBackOff\n\n"
        "A pod is stuck restarting because its container keeps crashing.\n"
        "Check: kubectl describe pod, kubectl logs, and recent events.\n"
    ),
    "imagepullbackoff.txt": (
        "Kubernetes ImagePullBackOff\n\n"
        "The node cannot pull the container image.\n"
        "Common causes include a wrong image name, a missing tag, or registry auth.\n"
        "Check: kubectl describe pod and the image pull events.\n"
    ),
}

for name, text in samples.items():
    path = DOCS_DIR / name
    if path.exists():
        print(f"Keeping existing {name}")
    else:
        path.write_text(text, encoding="utf-8")
        print(f"Wrote {name}")

In [ ]:
APP_PATH = Path("/content/rag_qwen_streamlit.py")
APP_PATH.write_text(
    r'''
import hashlib
import os
import sys
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import pysqlite3

    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except ImportError:
    pass

import chromadb
import streamlit as st
import torch
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS_DIR = Path(os.environ.get("RAG_DOCS_DIR", "docs"))
CHROMA_DIR = Path(os.environ.get("RAG_CHROMA_DIR", "/content/chroma_db_qwen3_embed"))
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
COLLECTION_NAME = "k8s_docs_qwen3_embed"

st.set_page_config(page_title="Qwen RAG", page_icon="📚")
st.title("Qwen RAG")
st.write("Ask a question over the files in `docs/`. Embeddings use Qwen3-Embedding-0.6B.")


def device_name() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def file_id(path: str) -> str:
    return Path(path).name


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


@st.cache_resource(show_spinner=False)
def load_embeddings():
    embed_device = device_name()
    return HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": embed_device},
        encode_kwargs={"normalize_embeddings": True},
        query_encode_kwargs={"prompt_name": "query", "normalize_embeddings": True},
    )


@st.cache_resource(show_spinner=False)
def load_chroma(_embeddings):
    CHROMA_DIR.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    return Chroma(
        client=client,
        collection_name=COLLECTION_NAME,
        embedding_function=_embeddings,
    )


@st.cache_resource(show_spinner=False)
def load_llm():
    llm_id = (
        "Qwen/Qwen2.5-1.5B-Instruct"
        if torch.cuda.is_available()
        else "Qwen/Qwen2.5-0.5B-Instruct"
    )
    hf_llm = HuggingFacePipeline.from_model_id(
        model_id=llm_id,
        task="text-generation",
        device_map="auto",
        model_kwargs={
            "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
        },
        pipeline_kwargs={
            "max_new_tokens": 256,
            "temperature": 0.1,
            "do_sample": True,
            "return_full_text": False,
        },
    )
    return ChatHuggingFace(llm=hf_llm), llm_id


def sync_vectorstore(embeddings):
    DOCS_DIR.mkdir(exist_ok=True)
    loader = DirectoryLoader(
        str(DOCS_DIR),
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    docs = loader.load()
    if not docs:
        return None, [], "No .txt files in docs/"

    current_hashes = {}
    for doc in docs:
        doc_id = file_id(doc.metadata["source"])
        doc_hash = sha256_text(doc.page_content)
        doc.metadata["doc_id"] = doc_id
        doc.metadata["doc_hash"] = doc_hash
        current_hashes[doc_id] = doc_hash

    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
    chunks = splitter.split_documents(docs)

    vectorstore = load_chroma(embeddings)

    stored = vectorstore.get(include=["metadatas"])
    stored_hashes = {}
    for meta in stored.get("metadatas") or []:
        if meta and meta.get("doc_id") and meta.get("doc_hash"):
            stored_hashes[meta["doc_id"]] = meta["doc_hash"]

    to_delete = [
        doc_id
        for doc_id, old_hash in stored_hashes.items()
        if current_hashes.get(doc_id) != old_hash
    ]
    for doc_id in to_delete:
        vectorstore.delete(where={"doc_id": doc_id})

    to_embed = {
        doc_id
        for doc_id, doc_hash in current_hashes.items()
        if stored_hashes.get(doc_id) != doc_hash
    }
    chunks_to_add = [chunk for chunk in chunks if chunk.metadata["doc_id"] in to_embed]
    if chunks_to_add:
        vectorstore.add_documents(chunks_to_add)
        status = (
            f"Embedded {len(chunks_to_add)} chunk(s) from "
            + ", ".join(sorted(to_embed))
        )
    else:
        status = "No document changes; reused the persisted Chroma index."

    return vectorstore, sorted(current_hashes), status


st.sidebar.header("Documents")
uploaded = st.sidebar.file_uploader(
    "Upload .txt files",
    type=["txt"],
    accept_multiple_files=True,
)
if uploaded:
    DOCS_DIR.mkdir(exist_ok=True)
    for item in uploaded:
        (DOCS_DIR / item.name).write_bytes(item.getvalue())
    st.sidebar.success(f"Saved {len(uploaded)} file(s) to docs/")

with st.spinner("Loading Qwen3-Embedding-0.6B (first run downloads the model)..."):
    embeddings = load_embeddings()
vectorstore, indexed_files, index_status = sync_vectorstore(embeddings)

st.sidebar.write(index_status)
if indexed_files:
    st.sidebar.write("Indexed files:")
    for name in indexed_files:
        st.sidebar.write(f"- {name}")

if vectorstore is None:
    st.warning("Add at least one .txt file to `docs/` (or upload one in the sidebar).")
    st.stop()

with st.spinner("Loading Qwen Instruct LLM (first run downloads the model)..."):
    llm, llm_id = load_llm()
st.caption(f"LLM: {llm_id} · embeddings: {EMBED_MODEL}")

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


prompt = ChatPromptTemplate.from_template(
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say you do not know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = st.text_input("Ask a question")
if question:
    with st.spinner("Retrieving and answering..."):
        retrieved = retriever.invoke(question)
        answer = rag_chain.invoke(question)
    st.subheader("Answer")
    st.write(answer)
    with st.expander("Retrieved chunks"):
        for i, chunk in enumerate(retrieved):
            st.markdown(f"**chunk {i} · {chunk.metadata.get('doc_id', 'unknown')}**")
            st.write(chunk.page_content)
'''.lstrip(),
    encoding="utf-8",
)
print("Wrote", APP_PATH)

In [ ]:
import os
import socket
import subprocess
import time

from google.colab import output
from google.colab.output import eval_js

subprocess.run(["pkill", "-f", "streamlit run /content/rag_qwen_streamlit.py"], check=False)
time.sleep(1)

proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/rag_qwen_streamlit.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true",
        "--server.enableCORS",
        "false",
        "--server.enableXsrfProtection",
        "false",
        "--browser.gatherUsageStats",
        "false",
    ],
    env={**os.environ},
)


def wait_for_port(port: int, timeout: int = 60) -> bool:
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket() as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(0.5)
    return False

if not wait_for_port(8501):
    raise RuntimeError("Streamlit did not start on port 8501. Re-run this cell.")

url = eval_js("google.colab.kernel.proxyPort(8501)")
print("Open the Streamlit app:", url)
print("The first page load downloads the Qwen models; wait for the spinners to finish.")
output.serve_kernel_port_as_iframe(8501, height=900)